# **Prerequisites**

In [1]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 2.5 MB/s eta 0:00:00


In [2]:
import requests
import pandas as pd
import time
import re
from tqdm import tqdm
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from bertopic import BERTopic
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [3]:
# ==============================
# GAME METADATA
# ==============================

def get_game_name(app_id):
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        if data[str(app_id)]["success"]:
            return data[str(app_id)]["data"]["name"]
    except:
        pass
    return f"Unknown Game ({app_id})"


# ==============================
# PLAYER STATS
# ==============================

def get_player_stats(app_id, game_name):
    url = f"https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={app_id}"
    try:
        r = requests.get(url)
        data = r.json()
        current_players = data["response"]["player_count"]
    except:
        current_players = "N/A"

    print(f"\n=== PLAYER SNAPSHOT: {game_name} ===")
    if current_players != "N/A":
        print(f"  Current Players:  {current_players:,}")
    else:
        print(f"  Current Players:  N/A")

# ==============================
# TEXT CLEANING
# ==============================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return text


# ==============================
# DATA COLLECTION
# ==============================

def fetch_reviews(app_id, source_name, num_reviews=1000):
    url = f"https://store.steampowered.com/appreviews/{app_id}"

    params = {
        "json": 1,
        "filter": "all",
        "language": "english",
        "day_range": 3650,
        "review_type": "all",
        "purchase_type": "all",
        "num_per_page": 100
    }

    reviews = []
    cursor = "*"

    with tqdm(total=num_reviews, desc=f"Fetching {source_name}") as pbar:
        while len(reviews) < num_reviews:
            params["cursor"] = cursor

            try:
                r = requests.get(url, params=params)
                data = r.json()
            except:
                time.sleep(2)
                continue

            if not data.get("reviews"):
                break

            for review in data["reviews"]:
                author = review.get("author", {})

                reviews.append({
                    "steamid": str(author.get("steamid")),
                    "review_text": review.get("review"),
                    "recommended": review.get("voted_up"),
                    "source": source_name
                })

                pbar.update(1)

                if len(reviews) >= num_reviews:
                    break

            cursor = data.get("cursor")
            time.sleep(1)

    df = pd.DataFrame(reviews)
    df["review_text"] = df["review_text"].apply(clean_text)

    return df

In [4]:
# ==============================
# SENTIMENT MODEL
# ==============================

def analyze_sentiment(df, label):
    X = df["review_text"].fillna("")
    y = df["recommended"].map({True: "Recommend", False: "Not Recommend"})

    vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
    X_vec = vectorizer.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(
        X_vec, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    print(f"\n=== SENTIMENT MODEL ({label}) ===")
    print(classification_report(y_test, preds))

    feature_names = vectorizer.get_feature_names_out()
    coef = model.coef_[0]

    n = 15
    top_negative_idx = coef.argsort()[:n]
    top_positive_idx = coef.argsort()[-n:][::-1]

    print(f"\nTop words driving NOT RECOMMEND:")
    for idx in top_negative_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    print(f"\nTop words driving RECOMMEND:")
    for idx in top_positive_idx:
        print(f"  {feature_names[idx]:<25} coef: {coef[idx]:.4f}")

    return model, vectorizer


# ==============================
# BERTOPIC
# ==============================

def analyze_topics_bertopic(df):
    print("\n=== TOPIC MODEL (BERTopic) ===")

    docs = df["review_text"].dropna().tolist()

    steam_stopwords = [
        "game", "play", "played", "playing",
        "like", "just", "get", "one", "will"
    ]

    vectorizer_model = CountVectorizer(
        stop_words=list(ENGLISH_STOP_WORDS) + steam_stopwords,
        ngram_range=(1, 2),
        min_df=5
    )

    topic_model = BERTopic(
        vectorizer_model=vectorizer_model,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(docs)

    topic_info = topic_model.get_topic_info()

    print("\nTop Topics Found:")
    print(topic_info.head(15))

    print("\nSample Clean Topic Words:\n")
    for topic_id in topic_info["Topic"].head(10):
        if topic_id == -1:
            continue

        words = topic_model.get_topic(topic_id)
        words = [w[0] for w in words[:10]]
        print(f"Topic {topic_id}: {', '.join(words)}")

    return topic_model


# ==============================
# TOPIC x SENTIMENT BREAKDOWN
# ==============================

def topic_sentiment_breakdown(df, topic_model, sentiment_model, vectorizer):
    docs = df["review_text"].dropna().reset_index(drop=True)
    labels = df["recommended"].dropna().reset_index(drop=True)

    topics, _ = topic_model.transform(docs.tolist())

    X_vec = vectorizer.transform(docs)
    preds = sentiment_model.predict(X_vec)

    results = pd.DataFrame({
        "topic": topics,
        "predicted_sentiment": preds,
        "actual": labels.map({True: "Recommend", False: "Not Recommend"})
    })

    rows = []
    for topic_id, group in results[results["topic"] != -1].groupby("topic"):
        total = len(group)
        neg_pct = (group["predicted_sentiment"] == "Not Recommend").mean() * 100
        pos_pct = 100 - neg_pct
        words = topic_model.get_topic(topic_id)
        label = ", ".join([w[0] for w in words[:3]])
        rows.append((topic_id, label, total, pos_pct, neg_pct))

    negative_side = sorted([r for r in rows if r[4] > 50], key=lambda x: x[4], reverse=True)
    positive_side = sorted([r for r in rows if r[4] <= 50], key=lambda x: x[3], reverse=True)

    print("\n=== TOPIC-LEVEL SENTIMENT BREAKDOWN ===")

    print("\n--- MOST NEGATIVE ---")
    for r in negative_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👎 {r[4]:.1f}%  |  👍 {r[3]:.1f}%")

    print("\n--- MOST POSITIVE ---")
    for r in positive_side:
        print(f"  Topic {r[0]} [{r[1]}]")
        print(f"    Reviews: {r[2]}  |  👍 {r[3]:.1f}%  |  👎 {r[4]:.1f}%")

    return results

In [5]:
# ==============================
# FULL PIPELINE
# ==============================

def analyze_full_game(base_app_id, dlc_app_id=None, num_reviews=4000):
    game_name = get_game_name(base_app_id)
    print(f"\n===== BASE GAME: {game_name} (ID: {base_app_id}) =====")

    # Player snapshot at the top
    get_player_stats(base_app_id, game_name)

    base_df = fetch_reviews(base_app_id, "base", num_reviews)
    model, vectorizer = analyze_sentiment(base_df, "base")
    topic_model = analyze_topics_bertopic(base_df)
    breakdown = topic_sentiment_breakdown(base_df, topic_model, model, vectorizer)

    base_positive_rate = base_df["recommended"].mean()
    print("\n=== OVERALL BASE GAME SENTIMENT ===")
    print(f"Positive Rate: {base_positive_rate:.2f}")
    print(f"Negative Rate: {1 - base_positive_rate:.2f}")

    if dlc_app_id:
        dlc_name = get_game_name(dlc_app_id)
        print(f"\n===== DLC: {dlc_name} (ID: {dlc_app_id}) =====")

        # Player snapshot for DLC
        get_player_stats(dlc_app_id, dlc_name)

        dlc_df = fetch_reviews(dlc_app_id, "dlc", num_reviews // 2)
        analyze_sentiment(dlc_df, "dlc")
        analyze_topics_bertopic(dlc_df)
        topic_sentiment_breakdown(dlc_df, topic_model, model, vectorizer)

        dlc_positive_rate = dlc_df["recommended"].mean()
        print("\n=== OVERALL DLC SENTIMENT ===")
        print(f"Positive Rate: {dlc_positive_rate:.2f}")
        print(f"Negative Rate: {1 - dlc_positive_rate:.2f}")

        print("\n=== BASE vs DLC COMPARISON ===")
        print(f"Base positive rate:  {base_positive_rate:.2f}")
        print(f"DLC positive rate:   {dlc_positive_rate:.2f}")
        diff = dlc_positive_rate - base_positive_rate
        direction = "better" if diff > 0 else "worse"
        print(f"DLC was received {abs(diff)*100:.1f}% {direction} than the base game")

    return base_df, topic_model, model, vectorizer, breakdown

# **Tests of Recent Popular Games with App ID (AID)**

### **Elden Ring**

In [ ]:
# ===================================================================================
  # Elden Ring                      ->     AID: 1245620 (VERY Popular Game)
  # ARC Raiders                     ->     AID: 1808500 (Currently Most Played Game)

  # Call of Duty: Black Ops 7       ->     AID: 1938090 (NEGATIVE Reviews)
  # Shadow Of Doubt                 ->     AID: 1938090 (Niche Smaller Game)

  # Ninja Gaiden 4                  ->     AID: 2627260
  #  ''     ''   '' DLC             ->     AID: 4191490 (Test with DLC Added)

  # Marvel Rivals                   ->     AID: 2767030 (Popular Live Service Game)
  # HellDivers 2                    ->     AID: 553850  (Live Service Game)
# ===================================================================================


# ==============================
# BEGIN TESTS
# ==============================

# Elden Ring
data = analyze_full_game(1245620)


===== BASE GAME: ELDEN RING (ID: 1245620) =====

=== PLAYER SNAPSHOT: ELDEN RING ===
  Current Players:  38,227


Fetching base: 100%|██████████| 4000/4000 [01:06<00:00, 60.23it/s]
2026-05-08 14:18:07,708 - BERTopic - Embedding - Transforming documents to embeddings.



=== SENTIMENT MODEL (base) ===
               precision    recall  f1-score   support

Not Recommend       0.31      0.24      0.27        17
    Recommend       0.98      0.99      0.99       783

     accuracy                           0.97       800
    macro avg       0.65      0.61      0.63       800
 weighted avg       0.97      0.97      0.97       800


Top words driving NOT RECOMMEND:
  terrible                  coef: -3.6752
  fix                       coef: -3.3972
  save                      coef: -3.1139
  sekiro                    coef: -2.8899
  don                       coef: -2.6816
  fans                      coef: -2.5774
  insan                     coef: -2.5391
  launch                    coef: -2.5370
  instead                   coef: -2.3917
  taken                     coef: -2.3235
  away                      coef: -2.2996
  level                     coef: -2.2721
  issues                    coef: -2.2083
  garbage                   coef: -2.1921
  doesn      

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

2026-05-08 14:22:08,407 - BERTopic - Embedding - Completed ✓
2026-05-08 14:22:08,410 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-08 14:22:58,462 - BERTopic - Dimensionality - Completed ✓


### **Arc Raiders**

In [ ]:
# ARC Raiders
data = analyze_full_game(1808500)

### **Call of Duty: Black Ops 7**

In [ ]:
# COD Black Ops 7 (REVIEWED BOMBED)
data = analyze_full_game(1938090)

### **Shadow Of Doubt**

In [ ]:
# Shadow Of Doubt
data = analyze_full_game(986130)

### **Ninja Gaiden 4 w/ DLC**

In [ ]:
# Ninja Gaiden 4
# data = analyze_full_game(2627260)

# Ninja Gaiden 4 w/ DLC
data = analyze_full_game(2627260, dlc_app_id=4191490)

### **Marvel Rivals**

In [ ]:
# Marvel Rivals
data = analyze_full_game(2767030)

### **HellDivers 2**

In [ ]:
# HellDivers 2
data = analyze_full_game(553850)